In [0]:
# ============================================================
# 06_MANAGER_ASSIGNMENT — GLOBAL, CAPACITY-AWARE ASSIGNMENT
# Firmable | Rule-based only | Ground truth is NOT used
#
# Input:
#   workspace.firmable_v2_silver.manager_candidate_scores
#   workspace.firmable_v2_silver.employees_clean
#
# Output:
#   workspace.firmable_v2_silver.manager_assignments
#
# Design:
# - Process ranked candidates in order (rank 1, then rank 2, etc.).
# - Use a fallback only when the employee was not assigned at a
#   higher-ranked tier, including when the preferred manager is full.
# - Enforce manager capacity across all assignment rounds.
# - Exclude ambiguous employees and candidates below the score threshold.
# - Preserve the existing columns consumed by Notebook 07.
# - Add explanatory assignment metadata for auditability.
# - Ground truth is never read or used.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER = "workspace.firmable_v2_silver"
CANDIDATE_TABLE = f"{SILVER}.manager_candidate_scores"
EMPLOYEE_TABLE = f"{SILVER}.employees_clean"
OUTPUT_TABLE = f"{SILVER}.manager_assignments"

MIN_SCORE = 55
AMBIGUITY_MARGIN = 8

# ============================================================
# 1. LOAD INPUTS AND CHECK REQUIRED SCHEMAS
# ============================================================

candidates = spark.table(CANDIDATE_TABLE)
employees = spark.table(EMPLOYEE_TABLE)

required_candidate_columns = {
    "company_id",
    "employee_id",
    "candidate_manager_id",
    "candidate_score",
    "candidate_rank",
    "candidate_priority",
    "score_margin",
    "ambiguity_flag",
    "relationship_type",
    "relationship_tier",
    "manager_seniority",
    "score_evidence"
}
missing_candidate_columns = required_candidate_columns - set(candidates.columns)
if missing_candidate_columns:
    raise ValueError(
        "Notebook 06 stopped. Missing columns in candidate scores: "
        f"{sorted(missing_candidate_columns)}"
    )

required_employee_columns = {
    "company_id",
    "employee_id",
    "inferred_seniority_level"
}
missing_employee_columns = required_employee_columns - set(employees.columns)
if missing_employee_columns:
    raise ValueError(
        "Notebook 06 stopped. Missing columns in employees_clean: "
        f"{sorted(missing_employee_columns)}"
    )

if candidates.limit(1).count() == 0:
    raise ValueError("No scored candidates found. Run Notebook 05 first.")

# Use one canonical employee row per company/employee.
employee_keys = employees.select(
    "company_id", "employee_id", "inferred_seniority_level"
)

duplicate_employee_keys = (
    employee_keys
    .groupBy("company_id", "employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
if duplicate_employee_keys:
    raise ValueError(
        f"employees_clean contains {duplicate_employee_keys} duplicate "
        "company/employee keys. Resolve these before assignment."
    )

# ============================================================
# 2. PRE-ASSIGNMENT INTEGRITY CHECKS
# ============================================================

duplicate_candidate_pairs = (
    candidates
    .groupBy("company_id", "employee_id", "candidate_manager_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_candidate_ids = candidates.filter(
    F.col("company_id").isNull() |
    F.col("employee_id").isNull() |
    F.col("candidate_manager_id").isNull() |
    (F.col("employee_id") == F.col("candidate_manager_id"))
).count()

invalid_scores = candidates.filter(
    F.col("candidate_score").isNull() |
    (F.col("candidate_score") < 0) |
    (F.col("candidate_score") > 100)
).count()

invalid_ranks = candidates.filter(
    F.col("candidate_rank").isNull() |
    (F.col("candidate_rank") < 1)
).count()

invalid_ambiguity_values = candidates.filter(
    F.col("ambiguity_flag").isNull()
).count()

# Verify that each candidate manager exists in the same company and
# that their actual Silver seniority is more senior than the employee.
employee_levels = employee_keys.select(
    "company_id",
    F.col("employee_id").alias("checked_employee_id"),
    F.col("inferred_seniority_level").alias("actual_employee_seniority")
)

manager_levels = employee_keys.select(
    "company_id",
    F.col("employee_id").alias("checked_manager_id"),
    F.col("inferred_seniority_level").alias("actual_manager_seniority")
)

candidate_reference_check = (
    candidates.alias("c")
    .join(
        manager_levels.alias("m"),
        (F.col("c.company_id") == F.col("m.company_id")) &
        (F.col("c.candidate_manager_id") == F.col("m.checked_manager_id")),
        "left"
    )
    .join(
        employee_levels.alias("e"),
        (F.col("c.company_id") == F.col("e.company_id")) &
        (F.col("c.employee_id") == F.col("e.checked_employee_id")),
        "left"
    )
)

missing_employee_references = candidate_reference_check.filter(
    F.col("e.actual_employee_seniority").isNull()
).count()

missing_manager_references = candidate_reference_check.filter(
    F.col("m.actual_manager_seniority").isNull()
).count()

invalid_seniority_relationships = candidate_reference_check.filter(
    F.col("m.actual_manager_seniority") >=
    F.col("e.actual_employee_seniority")
).count()

print("========== PRE-ASSIGNMENT VALIDATION ==========")
print("Candidate rows:", candidates.count())
print("Duplicate employee-manager pairs:", duplicate_candidate_pairs)
print("Invalid candidate IDs / self relationships:", invalid_candidate_ids)
print("Invalid candidate scores:", invalid_scores)
print("Invalid candidate ranks:", invalid_ranks)
print("Null ambiguity flags:", invalid_ambiguity_values)
print("Missing employee references:", missing_employee_references)
print("Missing manager references:", missing_manager_references)
print("Invalid seniority relationships:", invalid_seniority_relationships)

if any([
    duplicate_candidate_pairs,
    invalid_candidate_ids,
    invalid_scores,
    invalid_ranks,
    invalid_ambiguity_values,
    missing_employee_references,
    missing_manager_references,
    invalid_seniority_relationships
]):
    raise ValueError(
        "Pre-assignment validation failed. manager_assignments was NOT overwritten."
    )

# ============================================================
# 3. BUILD EMPLOYEE-LEVEL CANDIDATE SUMMARY
# ============================================================

candidate_summary = (
    candidates
    .groupBy("company_id", "employee_id")
    .agg(
        F.count("*").alias("candidate_count"),
        F.max("candidate_score").alias("best_candidate_score"),
        F.max(F.col("ambiguity_flag").cast("int")).alias("is_ambiguous"),
        F.max("candidate_rank").alias("maximum_candidate_rank"),
        F.sum(
            (F.col("candidate_score") >= MIN_SCORE).cast("int")
        ).alias("candidates_meeting_minimum_score")
    )
)

# An employee flagged ambiguous by Notebook 05 is held out of automatic
# assignment. The score margin is a top-vs-second comparison for that employee.
eligible_candidates = candidates.filter(
    (F.col("candidate_score") >= MIN_SCORE) &
    (F.col("ambiguity_flag") == F.lit(False))
)

# ============================================================
# 4. DEFINE MANAGER CAPACITY
# Capacity is a hard maximum, not a target.
# Keep the established project capacity policy.
# ============================================================

eligible_candidates = eligible_candidates.withColumn(
    "manager_capacity",
    F.when(F.col("manager_seniority") <= 0, 8)
     .when(F.col("manager_seniority") <= 7, 10)
     .when(F.col("manager_seniority") <= 9, 8)
     .otherwise(6)
)

# Use the maximum available candidate rank rather than hard-coding
# three rounds. This supports future candidate tiers without code edits.
max_candidate_rank = candidates.agg(
    F.max("candidate_rank").alias("max_rank")
).first()["max_rank"]

if max_candidate_rank is None or max_candidate_rank < 1:
    raise ValueError("No valid candidate ranks found.")

print("Minimum assignment score:", MIN_SCORE)
print("Ambiguity margin:", AMBIGUITY_MARGIN)
print("Assignment rounds available:", max_candidate_rank)

# ============================================================
# 5. GLOBAL CAPACITY-AWARE, RANKED ASSIGNMENT
#
# Round 1: each employee competes for their rank-1 candidate.
# Round 2: only employees not assigned in round 1 try rank 2.
# Round 3+: same pattern for subsequent fallback candidates.
#
# Each manager's remaining capacity is recalculated after every round.
# A fallback is therefore considered only if the preferred candidate
# was unavailable, ambiguous/ineligible, below threshold, or full.
# ============================================================

all_employee_keys = employee_keys.select(
    "company_id", "employee_id"
).dropDuplicates()

accepted_assignments = None
round_diagnostics = []

for round_number in range(1, int(max_candidate_rank) + 1):

    if accepted_assignments is None:
        remaining_employees = all_employee_keys
        prior_load = None
    else:
        already_assigned = accepted_assignments.select(
            "company_id", "employee_id"
        ).dropDuplicates()

        remaining_employees = all_employee_keys.join(
            already_assigned,
            ["company_id", "employee_id"],
            "left_anti"
        )

        prior_load = (
            accepted_assignments
            .groupBy("company_id", "candidate_manager_id")
            .agg(F.count("*").alias("current_direct_reports"))
        )

    round_candidates = (
        eligible_candidates
        .filter(F.col("candidate_rank") == round_number)
        .join(
            remaining_employees,
            ["company_id", "employee_id"],
            "inner"
        )
    )

    if prior_load is None:
        round_candidates = round_candidates.withColumn(
            "current_direct_reports", F.lit(0)
        )
    else:
        round_candidates = (
            round_candidates
            .join(
                prior_load,
                ["company_id", "candidate_manager_id"],
                "left"
            )
            .withColumn(
                "current_direct_reports",
                F.coalesce(F.col("current_direct_reports"), F.lit(0))
            )
        )

    round_candidates = (
        round_candidates
        .withColumn(
            "available_capacity",
            F.col("manager_capacity") - F.col("current_direct_reports")
        )
        .filter(F.col("available_capacity") > 0)
    )

    # If multiple employees compete for a manager's remaining slots,
    # prioritize stronger evidence, then larger score margin, then
    # candidate priority and stable employee ID for deterministic output.
    manager_competition_window = (
        Window
        .partitionBy("company_id", "candidate_manager_id")
        .orderBy(
            F.col("candidate_score").desc(),
            F.col("score_margin").desc_nulls_last(),
            F.col("candidate_priority").asc(),
            F.col("employee_id").asc()
        )
    )

    round_accepted = (
        round_candidates
        .withColumn(
            "manager_slot_rank",
            F.row_number().over(manager_competition_window)
        )
        .filter(
            F.col("manager_slot_rank") <= F.col("available_capacity")
        )
        .withColumn("assignment_round", F.lit(round_number))
        .withColumn("assignment_status", F.lit("ASSIGNED"))
        .withColumn(
            "assignment_method",
            F.when(F.col("candidate_rank") == 1, "PRIMARY")
             .otherwise("FALLBACK")
        )
        .withColumn(
            "assignment_reason",
            F.when(
                F.col("candidate_rank") == 1,
                "HIGHEST_RANKED_ELIGIBLE_CANDIDATE"
            ).otherwise("NEXT_RANKED_CANDIDATE_USED")
        )
        .select(
            "company_id",
            "employee_id",
            "candidate_manager_id",
            "candidate_score",
            "candidate_rank",
            "candidate_priority",
            "score_margin",
            "relationship_type",
            "relationship_tier",
            "score_evidence",
            "manager_seniority",
            "manager_capacity",
            "assignment_round",
            "assignment_status",
            "assignment_method",
            "assignment_reason"
        )
    )

    accepted_this_round = round_accepted.count()
    round_diagnostics.append((round_number, accepted_this_round))

    if accepted_assignments is None:
        accepted_assignments = round_accepted
    else:
        accepted_assignments = accepted_assignments.unionByName(round_accepted)

    print(
        f"Round {round_number}: accepted {accepted_this_round} assignments"
    )

if accepted_assignments is None:
    raise ValueError("Assignment process did not produce an assignment DataFrame.")

# ============================================================
# 6. CREATE ONE OUTPUT ROW FOR EVERY EMPLOYEE
# ============================================================

assignment_output = (
    all_employee_keys
    .join(
        accepted_assignments,
        ["company_id", "employee_id"],
        "left"
    )
    .join(
        candidate_summary,
        ["company_id", "employee_id"],
        "left"
    )
    .withColumn(
        "assignment_status",
        F.when(
            F.col("candidate_manager_id").isNotNull(),
            F.lit("ASSIGNED")
        )
        .when(
            F.col("is_ambiguous") == 1,
            F.lit("ABSTAINED_AMBIGUOUS")
        )
        .when(
            F.col("candidate_count").isNotNull() &
            (F.col("candidates_meeting_minimum_score") == 0),
            F.lit("ABSTAINED_LOW_SCORE")
        )
        .when(
            F.col("candidate_count").isNotNull(),
            F.lit("UNASSIGNED_CAPACITY_OR_ELIGIBILITY")
        )
        .otherwise(F.lit("NO_CANDIDATE_OR_ROOT"))
    )
    .withColumn(
        "assignment_method",
        F.when(F.col("candidate_manager_id").isNotNull(), F.col("assignment_method"))
         .otherwise("ABSTAIN")
    )
    .withColumn(
        "assignment_reason",
        F.when(
            F.col("candidate_manager_id").isNotNull(),
            F.col("assignment_reason")
        )
        .when(
            F.col("is_ambiguous") == 1,
            F.lit("TOP_CANDIDATES_WITHIN_AMBIGUITY_MARGIN")
        )
        .when(
            F.col("candidate_count").isNull(),
            F.lit("NO_ELIGIBLE_CANDIDATE_GENERATED")
        )
        .when(
            F.col("candidates_meeting_minimum_score") == 0,
            F.lit("ALL_CANDIDATES_BELOW_MINIMUM_SCORE")
        )
        .otherwise(
            F.lit("ELIGIBLE_CANDIDATES_EXHAUSTED_MANAGER_CAPACITY")
        )
    )
    .withColumn(
        "relationship_tier",
        F.coalesce(F.col("relationship_tier"), F.lit("UNASSIGNED"))
    )
)

# ============================================================
# 7. FINAL INTEGRITY CHECKS
# ============================================================

total_employees = all_employee_keys.count()
output_count = assignment_output.count()

duplicate_employee_assignments = (
    assignment_output
    .groupBy("company_id", "employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assigned_rows = assignment_output.filter(
    F.col("candidate_manager_id").isNotNull()
)

assigned_count = assigned_rows.count()
self_relationships = assigned_rows.filter(
    F.col("employee_id") == F.col("candidate_manager_id")
).count()

# Recheck manager existence and seniority using employees_clean.
final_seniority_check = (
    assigned_rows.alias("a")
    .join(
        employee_levels.alias("e"),
        (F.col("a.company_id") == F.col("e.company_id")) &
        (F.col("a.employee_id") == F.col("e.checked_employee_id")),
        "inner"
    )
    .join(
        manager_levels.alias("m"),
        (F.col("a.company_id") == F.col("m.company_id")) &
        (F.col("a.candidate_manager_id") == F.col("m.checked_manager_id")),
        "inner"
    )
)

invalid_final_seniority = final_seniority_check.filter(
    F.col("m.actual_manager_seniority") >=
    F.col("e.actual_employee_seniority")
).count()

# Verify manager capacity across the complete final assignment set.
capacity_violations = (
    assigned_rows
    .groupBy("company_id", "candidate_manager_id")
    .agg(
        F.count("*").alias("actual_direct_reports"),
        F.max("manager_capacity").alias("allowed_capacity")
    )
    .filter(F.col("actual_direct_reports") > F.col("allowed_capacity"))
    .count()
)

missing_manager_refs_after_assignment = (
    assigned_rows
    .join(
        manager_levels.select(
            "company_id", "checked_manager_id"
        ),
        (assigned_rows.company_id == manager_levels.company_id) &
        (assigned_rows.candidate_manager_id == manager_levels.checked_manager_id),
        "left_anti"
    )
    .count()
)

print("\n========== FINAL ASSIGNMENT VALIDATION ==========")
print("Total employees:", total_employees)
print("Output rows:", output_count)
print("Assigned employees:", assigned_count)
print("Unassigned employees:", total_employees - assigned_count)
print("Duplicate employee assignments:", duplicate_employee_assignments)
print("Self relationships:", self_relationships)
print("Invalid seniority relationships:", invalid_final_seniority)
print("Missing manager references:", missing_manager_refs_after_assignment)
print("Manager capacity violations:", capacity_violations)

if any([
    output_count != total_employees,
    duplicate_employee_assignments > 0,
    self_relationships > 0,
    invalid_final_seniority > 0,
    missing_manager_refs_after_assignment > 0,
    capacity_violations > 0
]):
    raise ValueError(
        "Final assignment validation failed. manager_assignments was NOT overwritten."
    )

# ============================================================
# 8. PUBLISH ASSIGNMENTS
# Existing Notebook 07 columns are preserved:
# candidate_manager_id, candidate_score, relationship_type,
# assignment_status, assignment_round.
# Additional audit columns are additive.
# ============================================================

(
    assignment_output
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE)
)

# ============================================================
# 9. REPORT RESULTS
# ============================================================

print("\n========== MANAGER ASSIGNMENT COMPLETED ==========")
print(f"Total employees: {total_employees}")
print(f"Assigned employees: {assigned_count}")
print(f"Unassigned employees: {total_employees - assigned_count}")
print(f"Assignment rate: {assigned_count / total_employees * 100:.2f}%")
print(f"Minimum candidate score: {MIN_SCORE}")
print(f"Ambiguity margin: {AMBIGUITY_MARGIN}")
print(f"Assignment rounds executed: {max_candidate_rank}")

print("\nASSIGNMENT STATUS")
spark.table(OUTPUT_TABLE).groupBy(
    "assignment_status"
).count().orderBy(
    "assignment_status"
).show(truncate=False)

print("\nASSIGNMENTS BY ROUND")
spark.table(OUTPUT_TABLE).filter(
    F.col("candidate_manager_id").isNotNull()
).groupBy(
    "assignment_round"
).count().orderBy(
    "assignment_round"
).show(truncate=False)

print("\nASSIGNMENTS BY METHOD")
spark.table(OUTPUT_TABLE).groupBy(
    "assignment_method"
).count().orderBy(
    "assignment_method"
).show(truncate=False)

print("\nASSIGNMENTS BY RELATIONSHIP TIER")
spark.table(OUTPUT_TABLE).filter(
    F.col("candidate_manager_id").isNotNull()
).groupBy(
    "relationship_tier",
    "relationship_type"
).count().orderBy(
    "relationship_tier",
    "relationship_type"
).show(100, truncate=False)

print("\nCAPACITY / INTEGRITY")
print("Capacity violations:", capacity_violations)
print("Duplicate employee assignments:", duplicate_employee_assignments)
print("Self relationships:", self_relationships)
print("Invalid seniority relationships:", invalid_final_seniority)
print("Missing manager references:", missing_manager_refs_after_assignment)

print(f"\nOUTPUT TABLE: {OUTPUT_TABLE}")
print("candidate_score is a rule-based score, NOT a probability.")
print("Ground truth was not read or used.")
print("ALL ASSIGNMENT INTEGRITY CHECKS PASSED")


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.firmable_v2_silver.manager_assignments
DEEP CLONE workspace.firmable_v2_silver.manager_assignments_pre_reconciliation;